# 07 — Hard visual/semantic disagreement pairs

The benchmark is deliberately constructed **before** fusion evaluation. Its purpose is to expose cases where surface visual neighbourhoods and iconographic neighbourhoods disagree.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").is_dir() and (ROOT.parent / "src").is_dir():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import numpy as np
from caypollard.benchmarks.hard_pairs import HardPairThresholds, mine_hard_pairs
from caypollard.embeddings.store import EmbeddingTable

In [ ]:
parents = {
    "A": set(), "A1": {"A"}, "A2": {"A"},
    "B": set(), "B1": {"B"}, "B2": {"B"},
}
records = [
    {"id": "a1", "iconclass": ["A1"]},
    {"id": "a2", "iconclass": ["A1"]},
    {"id": "a3", "iconclass": ["A2"]},
    {"id": "b1", "iconclass": ["B1"]},
    {"id": "b2", "iconclass": ["B1"]},
    {"id": "b3", "iconclass": ["B2"]},
]
table = EmbeddingTable(
    ids=tuple(row["id"] for row in records),
    vectors=np.asarray([
        [1.0, 0.0], [-1.0, 0.0], [-0.9, 0.1],
        [0.99, 0.01], [0.9, 0.1], [0.0, 1.0],
    ], dtype=np.float32),
    metadata={"fixture": True},
)
thresholds = HardPairThresholds(
    visual_close_min=0.8, visual_distant_max=0.0,
    semantic_close_min=0.5, semantic_distant_max=0.0,
)

In [ ]:
pairs, metadata = mine_hard_pairs(
    table, records, parents, thresholds,
    query_ids=["a1", "a2", "b1"],
    visual_top_k=3, random_distant_per_query=4, max_per_class=10, seed=3,
)
metadata

In [ ]:
[(p.pair_class, p.query_id, p.candidate_id, round(p.visual_similarity, 3), round(p.semantic_similarity, 3)) for p in pairs]

## Full benchmark rule

Real visual thresholds are not chosen by looking at test failures. They are calibrated from a fixed validation random-pair distribution (95th percentile for “close”, median for “distant”), persisted, and then applied unchanged to the test split.